# Question 5: Algorithm Design Strategies for Logistics

## Introduction

A logistics company optimizes delivery routes across Rwanda. This notebook implements and **cross-validates** three classic algorithm design strategies — Greedy, Dynamic Programming, and Backtracking — on the same city network, then compares them quantitatively rather than just by complexity class on paper.

## (a) Greedy Algorithm for Route Selection

The **nearest-neighbour** heuristic always travels next to the closest unvisited city. It is fast (O(n²)) but, being a greedy strategy, has no guarantee of producing the globally optimal route.

In [1]:
city_names = ["Kigali", "Musanze", "Rubavu", "Huye"]
dist = [
    [0, 60, 95, 135],
    [60, 0, 35, 190],
    [95, 35, 0, 220],
    [135, 190, 220, 0],
]

def greedy_route(distance_matrix, start, city_names):
    n = len(distance_matrix)
    visited = [False] * n
    visited[start] = True
    route = [start]
    total_cost = 0
    current = start
    for _ in range(n - 1):
        nearest, nearest_cost = None, float('inf')
        for city in range(n):
            if not visited[city] and distance_matrix[current][city] < nearest_cost:
                nearest = city
                nearest_cost = distance_matrix[current][city]
        visited[nearest] = True
        route.append(nearest)
        total_cost += nearest_cost
        current = nearest
    return [city_names[i] for i in route], total_cost

greedy_path, greedy_cost = greedy_route(dist, 0, city_names)
print("Greedy route:", " -> ".join(greedy_path))
print("Greedy one-way cost:", greedy_cost, "km")

Greedy route: Kigali -> Musanze -> Rubavu -> Huye
Greedy one-way cost: 315 km


## (b) Dynamic Programming for Cost Optimization

The Held–Karp style DP finds the **provably optimal** round-trip route by caching subproblem results (which subset of cities has been visited, and where we currently are), avoiding the combinatorial blow-up of naive brute force while still guaranteeing optimality.

In [2]:
def min_delivery_cost(distance_matrix, start, must_visit):
    n = len(must_visit)
    full_set = (1 << n) - 1
    memo = {}
    def dp(pos, visited_mask):
        if visited_mask == full_set:
            return distance_matrix[pos][start]
        if (pos, visited_mask) in memo:
            return memo[(pos, visited_mask)]
        best = float('inf')
        for nxt in range(n):
            if not (visited_mask & (1 << nxt)):
                cost = distance_matrix[pos][must_visit[nxt]] + dp(must_visit[nxt], visited_mask | (1 << nxt))
                best = min(best, cost)
        memo[(pos, visited_mask)] = best
        return best
    return dp(start, 0)

dp_cost = min_delivery_cost(dist, 0, [1, 2, 3])
print("Dynamic Programming optimal ROUND-TRIP cost:", dp_cost, "km")

Dynamic Programming optimal ROUND-TRIP cost: 450 km


In [3]:
# ---- Verify the DP result against brute-force ground truth ----
from itertools import permutations

def brute_force_round_trip(dist, start):
    n = len(dist)
    others = [c for c in range(n) if c != start]
    best = float('inf')
    for perm in permutations(others):
        route = [start] + list(perm)
        cost = sum(dist[route[i]][route[i+1]] for i in range(len(route)-1)) + dist[route[-1]][start]
        best = min(best, cost)
    return best

bf_cost = brute_force_round_trip(dist, 0)
print("Brute-force ground truth:", bf_cost, "km")
assert dp_cost == bf_cost, f"DP result {dp_cost} disagrees with brute force {bf_cost}!"
print("DP result MATCHES brute-force optimum ✔ — the DP solution is provably correct")

Brute-force ground truth: 450 km
DP result MATCHES brute-force optimum ✔ — the DP solution is provably correct


## (c) Backtracking for Constrained Route Selection

Backtracking explores the full search tree of possible routes but **prunes** any partial route that already exceeds a budget constraint — useful when the company needs *all* routes satisfying a business rule (e.g. a fuel/cost ceiling), not just the single best one.

In [4]:
def constrained_routes(city_names, distance_matrix, max_budget):
    results = []
    def backtrack(path, visited, cost):
        if cost > max_budget:
            return  # prune
        if len(path) == len(city_names):
            results.append((path, cost))
            return
        last = path[-1]
        for nxt in range(len(city_names)):
            if nxt not in visited:
                backtrack(path + [nxt], visited | {nxt}, cost + distance_matrix[last][nxt])
    backtrack([0], {0}, 0)
    return results

valid_routes = constrained_routes(city_names, dist, max_budget=420)
print(f"Routes found within 420 km budget: {len(valid_routes)}\n")
for path, cost in sorted(valid_routes, key=lambda x: x[1]):
    named = [city_names[i] for i in path]
    print(f"  {' -> '.join(named):45} cost = {cost} km")

cheapest = min(valid_routes, key=lambda x: x[1])
assert [city_names[i] for i in cheapest[0]] == greedy_path
print("\nCheapest backtracking route matches the Greedy route ✔ (expected for this small instance)")

Routes found within 420 km budget: 4

  Kigali -> Musanze -> Rubavu -> Huye           cost = 315 km
  Kigali -> Rubavu -> Musanze -> Huye           cost = 320 km
  Kigali -> Huye -> Musanze -> Rubavu           cost = 360 km
  Kigali -> Huye -> Rubavu -> Musanze           cost = 390 km

Cheapest backtracking route matches the Greedy route ✔ (expected for this small instance)


## (d) Performance Comparison

| Strategy | Optimal Guarantee? | Time Complexity | Characteristics |
|---|---|---|---|
| Greedy | No | O(n²) | Fast, simple, locally optimal only |
| Dynamic Programming | Yes | O(n² × 2ⁿ) | Exact, uses memoization, exponential in n |
| Backtracking | Yes (if exhaustive) | O(n!) worst case | Explores all options, pruning helps but worst case is factorial |

**Empirical comparison** on this 4-city instance:

In [5]:
import time

def time_it(fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - start

_, t_greedy = time_it(greedy_route, dist, 0, city_names)
_, t_dp = time_it(min_delivery_cost, dist, 0, [1, 2, 3])
_, t_bt = time_it(constrained_routes, city_names, dist, 420)

print(f"{'Strategy':22} | {'Time (s)':>12} | {'Cost found (km)':>16}")
print(f"{'Greedy':22} | {t_greedy:>12.8f} | {greedy_cost:>16}")
print(f"{'Dynamic Programming':22} | {t_dp:>12.8f} | {dp_cost:>16}")
print(f"{'Backtracking (best)':22} | {t_bt:>12.8f} | {cheapest[1]:>16}")

print("\nConclusion: Greedy is fastest but only guarantees a locally-optimal (here, also globally optimal)")
print("one-way route. DP guarantees the true round-trip optimum at extra computational cost.")
print("Backtracking finds every feasible route under the budget constraint, which neither Greedy nor DP do.")

Strategy               |     Time (s) |  Cost found (km)
Greedy                 |   0.00000912 |              315
Dynamic Programming    |   0.00001659 |              450
Backtracking (best)    |   0.00001501 |              315

Conclusion: Greedy is fastest but only guarantees a locally-optimal (here, also globally optimal)
one-way route. DP guarantees the true round-trip optimum at extra computational cost.
Backtracking finds every feasible route under the budget constraint, which neither Greedy nor DP do.
